In [2]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.models.popularity import add_recent_counts, smoothed_ctr

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"
RES.mkdir(parents=True, exist_ok=True)

HOURS = (12, 24)
ALPHAS = (5, 20, 100)

In [3]:
imp = spark.read.parquet(str(OUT / "impressions_long"))

# Solo lo necesario para evaluar en val: el 14 nov + las 24 h previas (13 nov).
# El límite se escribe como texto para que Spark lo interprete en la zona horaria
# de la sesión (UTC), igual que los timestamps de los datos.
desde = F.to_timestamp(F.lit("2019-11-13 00:00:00"))

base = (imp.filter(F.col("periodo").isin("train", "val"))   # test no se lee
           .filter(F.col("time") >= desde)
           .select("impression_id", "time", "news_id", "label", "periodo"))

pop = add_recent_counts(base, hours=HOURS).cache()

# Valor previo del suavizado: CTR global del periodo de entrenamiento (9–13 nov)
p0 = imp.filter("periodo = 'train'").agg(F.avg("label")).first()[0]
print(f"p0 (CTR global de train) = {p0:.4f}")
print("Filas para la ventana:", base.count())

val = pop.filter("periodo = 'val'")
val.select("impression_id", "time", "news_id", "label",
           "expos_12h", "clics_12h", "expos_24h", "clics_24h").show(8)

p0 (CTR global de train) = 0.0410
Filas para la ventana: 2454102
+-------------+-------------------+-------+-----+---------+---------+---------+---------+
|impression_id|               time|news_id|label|expos_12h|clics_12h|expos_24h|clics_24h|
+-------------+-------------------+-------+-----+---------+---------+---------+---------+
|       142777|2019-11-14 04:27:31| N10088|    0|        1|        0|       10|        0|
|        59220|2019-11-14 05:18:26| N10088|    0|        2|        0|        8|        0|
|         8208|2019-11-14 06:21:14| N10088|    1|        3|        0|        9|        0|
|        49333|2019-11-14 07:12:20| N10088|    0|        4|        1|        9|        1|
|        57836|2019-11-14 08:12:13| N10088|    0|        5|        1|        9|        1|
|        89818|2019-11-14 08:15:59| N10088|    0|        5|        1|       10|        1|
|        11450|2019-11-14 10:17:23| N10088|    1|        6|        1|       11|        1|
|        27630|2019-11-14 11:37:54|

In [4]:
fila = val.filter("expos_24h > 0").orderBy(F.rand(3)).first()
t, n = fila["time"], fila["news_id"]

manual = (imp.filter((F.col("news_id") == n) &
                     (F.col("time") >= F.lit(t) - F.expr("INTERVAL 24 HOURS")) &
                     (F.col("time") < F.lit(t)))
             .agg(F.count("*").alias("expos"), F.sum("label").alias("clics"))
             .first())

print(f"Noticia {n} en {t}")
print(f"  Ventana:  expos_24h = {fila['expos_24h']},  clics_24h = {fila['clics_24h']}")
print(f"  Manual:   expos     = {manual['expos']},  clics     = {manual['clics']}")

Noticia N7494 en 2019-11-14 16:18:18
  Ventana:  expos_24h = 4278,  clics_24h = 90
  Manual:   expos     = 4278,  clics     = 90


In [5]:
val.agg(
    *[F.round(F.avg((F.col(f"expos_{h}h") == 0).cast("int")), 3).alias(f"cand_sin_expos_{h}h")
      for h in HOURS],
    *[F.round(F.sum(F.col("label") * (F.col(f"expos_{h}h") == 0).cast("int")) / F.sum("label"), 3)
      .alias(f"clics_sin_expos_{h}h") for h in HOURS],
).show()

+------------------+------------------+-------------------+-------------------+
|cand_sin_expos_12h|cand_sin_expos_24h|clics_sin_expos_12h|clics_sin_expos_24h|
+------------------+------------------+-------------------+-------------------+
|             0.003|             0.003|              0.004|              0.003|
+------------------+------------------+-------------------+-------------------+



In [6]:
variantes = {"aleatorio": F.rand(1)}

# Candidatos a baseline principal: CTR suavizado, rejilla h × alpha
for h in HOURS:
    for a in ALPHAS:
        variantes[f"pop_ctr_{h}h_a{a}"] = smoothed_ctr(h, a, p0)

# Referencias
for h in HOURS:
    variantes[f"pop_clics_{h}h"] = F.col(f"clics_{h}h")
    variantes[f"pop_expos_{h}h"] = F.col(f"expos_{h}h")

filas = []
for nombre, expr in variantes.items():
    per_imp, res = evaluate(val.withColumn("score", expr))
    per_imp.unpersist()          # libera memoria entre variantes
    filas.append({"modelo": nombre, **res})
    print(f"{nombre:22s} AUC = {res['auc']:.4f}")

tabla_val = (pd.DataFrame(filas)[["modelo", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
             .sort_values("auc", ascending=False)
             .reset_index(drop=True))
tabla_val.round(4)

aleatorio              AUC = 0.5001
pop_ctr_12h_a5         AUC = 0.6458
pop_ctr_12h_a20        AUC = 0.6468
pop_ctr_12h_a100       AUC = 0.6448
pop_ctr_24h_a5         AUC = 0.6468
pop_ctr_24h_a20        AUC = 0.6475
pop_ctr_24h_a100       AUC = 0.6453
pop_clics_12h          AUC = 0.5521
pop_expos_12h          AUC = 0.4586
pop_clics_24h          AUC = 0.5541
pop_expos_24h          AUC = 0.4606


,modelo,auc,mrr,ndcg5,ndcg10,impresiones
0,pop_ctr_24h_a20,0.6475,0.2786,0.2936,0.3567,30270
1,pop_ctr_12h_a20,0.6468,0.2788,0.2937,0.3569,30270
2,pop_ctr_24h_a5,0.6468,0.2798,0.2949,0.3580,30270
3,pop_ctr_12h_a5,0.6458,0.2798,0.2949,0.3580,30270
4,pop_ctr_24h_a100,0.6453,0.2753,0.2891,0.3529,30270
5,pop_ctr_12h_a100,0.6448,0.2748,0.2885,0.3524,30270
6,pop_clics_24h,0.5541,0.2323,0.2369,0.2967,30270
7,pop_clics_12h,0.5521,0.2324,0.2376,0.2964,30270
8,aleatorio,0.5001,0.2042,0.2048,0.2640,30270
9,pop_expos_24h,0.4606,0.1816,0.1795,0.2400,30270


In [7]:
ctr = tabla_val[tabla_val["modelo"].str.startswith("pop_ctr")]
mejor = ctr.iloc[0]

print(f"Baseline principal (mejor CTR en val): {mejor['modelo']}")
print(f"  AUC val = {mejor['auc']:.4f}")

Baseline principal (mejor CTR en val): pop_ctr_24h_a20
  AUC val = 0.6475


In [8]:
(tabla_val.assign(periodo="val", dataset=cfg["dataset"], p0=p0)
          .to_csv(RES / "03_baseline_val.csv", index=False))
print("Guardado en", RES / "03_baseline_val.csv")

Guardado en c:\DevProjects\MMDS\RecSysMIND\results\metrics\03_baseline_val.csv


In [9]:
spark.stop()